<a href="https://colab.research.google.com/github/Youseef3550/flyrank-ml-yousef-assighn-1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring (Lane 2)**

I'm picking this lane because it maps directly onto the one dataset and pipeline I
already have running end-to-end: `content_refresh_anonymized.csv` plus the
`01_prepare_features → 05_build_pdf_report` scripts. The lane's job — ranking pages
for review with an evidence-backed reason code — is a decision-support problem, not
just a classification exercise, and the starter pipeline already shows a learned
model beating a hand-written rule at that exact task (baseline Precision@50 ≈ 0.24
vs. random forest ≈ 0.74, from `outputs/model_report.md`). That gap is the strongest
signal I have that there's real room to build something better over the next 7
weeks, so I'm starting here and will revisit by Week 4 if the warehouse data tells a
different story.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one content page (one row = one `content_id`, 90-day window).

**Decision it improves:** which pages a content reviewer with limited time should
look at first — refresh, expand, protect, prune, or just monitor.

**Who acts:** a content strategist / SEO reviewer at a FlyRank client who can
realistically touch maybe 20-50 pages a week, not all 30,000.

**Output:** a ranked queue of pages, each with a score and a reason code (e.g.
`stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`) the reviewer
can inspect before acting.

**Cost of a wrong recommendation:**
- *False positive* (flagged as needing review, but it's actually fine): wastes a
  reviewer's limited hours on a page that didn't need attention.
- *False negative* (a genuinely declining, high-traffic page never surfaces): the
  client keeps losing real visibility for weeks before anyone notices — a much more
  expensive miss than a false positive.

**Why data/ML helps, and why it's not just "train a model":** the real work is the
framing — defining what "stale" and "declining" honestly mean, choosing a window
that doesn't leak future information into features, building a baseline a human
can already trust, and only then checking whether a model earns its extra
complexity over that baseline. A human stays in the loop; nothing here fully
automates a content decision.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/Youseef3550/flyrank-ml-yousef-assighn-1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print(f"Total pages in starter slice: {len(df):,}")

print("\nTrend direction breakdown:")
print(df["trend_direction"].value_counts())

declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"\nPages declining WITH real demand (>=100 impressions/90d): {len(declining_with_demand):,} "
      f"({100*len(declining_with_demand)/len(df):.1f}% of the slice)")

low_ctr_visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"].between(1, 20)) & (df["ctr"] < 0.5)]
print(f"Visible pages (position 1-20, 500+ impressions) with CTR < 0.5%: {len(low_ctr_visible):,} "
      f"({100*len(low_ctr_visible)/len(df):.1f}% of the slice)")

print(f"\nMedian impressions_90d: {df['impressions_90d'].median():.0f}")
print(f"Median sessions_90d: {df['sessions_90d'].median():.0f}")
print(f"Distinct clients in slice: {df['client_id'].nunique()}")

Total pages in starter slice: 30,000

Trend direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Pages declining WITH real demand (>=100 impressions/90d): 13,152 (43.8% of the slice)
Visible pages (position 1-20, 500+ impressions) with CTR < 0.5%: 9,745 (32.5% of the slice)

Median impressions_90d: 731
Median sessions_90d: 7
Distinct clients in slice: 32


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** these are observed, historical patterns measured directly from
90-day windows in the anonymized starter slice. Any ranking I build is
decision-support — it narrows a reviewer's list, it doesn't make the call for them.

**What I can't say:** `trend_direction == "down"` is a same-window proxy label, not
a verified future decline — consolidation, seasonality, and plain noise can all
look like "decline" (lane guide, section 7), and I still have to rule those out
before treating a drop as real. I won't claim my work predicts Google's algorithm,
and I won't claim a refresh caused a recovery unless I run something closer to an
actual experiment. Language stays observed / measured / directional /
decision-support — never causal proof.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.